# S6: dev evaluation of the fine-tuned student (epoch2 vs epoch3)

**Before running:** Accelerator = GPU T4 x2, Internet = On, secrets `HF_TOKEN` **and** `API_DEEPSEEK` attached.
Inputs (attached by the push script): `agentic-distill-src` (code) and the output of kernel `agentic-distill-s5-train` (adapters).

14 dev tasks x 1 trial per checkpoint; epoch2 on GPU 0 and epoch3 on GPU 1 in parallel. Records in /kaggle/working/out/data/raw/.


In [ ]:
import sys, subprocess, os, glob
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
ADAPTERS = {e: os.path.dirname(p) for e in ("epoch1", "epoch2")
            for p in glob.glob(f"/kaggle/input/**/lora_ext/{e}/adapter_config.json", recursive=True)}
print("adapters:", ADAPTERS)
assert len(ADAPTERS) == 2, "attach the output of the agentic-distill-s8-train-more kernel"
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for label in ("HF_TOKEN", "API_DEEPSEEK"):
    os.environ[label] = s.get_secret(label)          # values never printed
print("secrets loaded:", [k for k in ("HF_TOKEN", "API_DEEPSEEK") if os.environ.get(k)])


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
# Select the code dataset explicitly: the attached training-kernel output also contains an OLD copy of the repo,
# and `find -name pyproject.toml | head -1` can pick that one (this broke the first S7 attempt).
D=$(find /kaggle/input -maxdepth 3 -type d -name agentic-distill-src | head -1)
echo "code dataset dir: $D"
Z=$(find "$D" -name agentic_distill_src.zip | head -1)
if [ -n "$Z" ]; then (cd /kaggle/working/repo && unzip -q "$Z"); else cp -r "$D"/. /kaggle/working/repo/; fi
cd /kaggle/working/repo
grep -q disable_sdpa_gqa src/agentic_distill/student_run.py
mkdir -p third_party && git clone -q https://github.com/sierra-research/tau2-bench third_party/tau2-bench
git -C third_party/tau2-bench checkout -q c06b07dd7c4d4ec61ba6ff9bcdf0b15dd7ef36d4
pip uninstall -y -q torchao
pip install -q -e third_party/tau2-bench audioop-lts pyyaml
pip install -q -U transformers peft accelerate
python -c "import tau2, transformers, peft, torch; print('ok; transformers', transformers.__version__, 'gpus', torch.cuda.device_count())"


In [ ]:
import json, threading
split = json.load(open("/kaggle/working/repo/data/splits/retail_split.json"))
DEV = split["dev_ids"][:8]   # smaller set (user, 2026-11-15): first 8 dev tasks; compare with epoch3 on the same 8 from the S6 records
print("dev tasks:", DEV)

def launch(epoch, gpu):
    env = {**os.environ, "PYTHONPATH": "src", "CUDA_VISIBLE_DEVICES": str(gpu)}
    cmd = ["python", "-m", "agentic_distill.student_run", "--tasks", *DEV, "--trials", "1", "--slice", "eval_student",
           "--label", f"s8-{epoch}", "--adapter", ADAPTERS[epoch], "--out-root", "/kaggle/working/out"]
    p = subprocess.Popen(cmd, cwd="/kaggle/working/repo", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def pump():
        with open(f"/kaggle/working/{epoch}.log", "w") as f:
            for line in p.stdout:
                f.write(line); f.flush()
                if line.startswith(("task=", "slice", "STOP", "Traceback")) or "Error" in line[:40]:
                    print(f"[{epoch}] {line}", end="", flush=True)
    t = threading.Thread(target=pump); t.start()
    return p, t

procs = [launch("epoch1", 0), launch("epoch2", 1)]
for p, t in procs:
    p.wait(); t.join()
print("exit codes:", [p.returncode for p, _ in procs])


In [ ]:
%%bash
cd /kaggle/working/repo
PYTHONPATH=src python -m agentic_distill.metrics \
  "s8e1=$(ls -d /kaggle/working/out/data/raw/eval_student_s8-epoch1_* | head -1)" \
  "s8e2=$(ls -d /kaggle/working/out/data/raw/eval_student_s8-epoch2_* | head -1)" 2>&1 | grep -v DEBUG
for f in /kaggle/working/epoch1.log /kaggle/working/epoch2.log; do echo "== $f"; grep -c "^task=" $f; tail -3 $f; done
